In [1]:
import numpy as np
import pandas as pd

In [2]:
# 1.Load data
train_tr = pd.read_csv('../data/train_transaction.csv')
train_id = pd.read_csv('../data/train_identity.csv')
test_tr  = pd.read_csv('../data/test_transaction.csv')
test_id  = pd.read_csv('../data/test_identity.csv')

In [3]:
print("train_transaction:", train_tr.shape)
print("train_identity  :", train_id.shape)
print("test_transaction :", test_tr.shape)
print("test_identity    :", test_id.shape)


train_transaction: (590540, 394)
train_identity  : (144233, 41)
test_transaction : (506691, 393)
test_identity    : (141907, 41)


In [4]:
# 2. Merge on TransactionID
train = train_tr.merge(train_id, on="TransactionID", how="left")
test  = test_tr.merge(test_id, on="TransactionID", how="left")

In [5]:
print("Merged train shape:", train.shape)
print("Merged test shape :", test.shape)


Merged train shape: (590540, 434)
Merged test shape : (506691, 433)


In [6]:
# 3. Basic target info
target_col = "isFraud"
assert target_col in train.columns, "isFraud not found in train data"
print(train[target_col].value_counts(normalize=True))

isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [7]:
# 4. Check missing values
na_train = train.isna().mean().sort_values(ascending=False)
na_test  = test.isna().mean().sort_values(ascending=False)

na_train.head(20)


id_24    0.991962
id_25    0.991310
id_07    0.991271
id_08    0.991271
id_21    0.991264
id_26    0.991257
id_27    0.991247
id_23    0.991247
id_22    0.991247
dist2    0.936284
D7       0.934099
id_18    0.923607
D13      0.895093
D14      0.894695
D12      0.890410
id_03    0.887689
id_04    0.887689
D6       0.876068
id_33    0.875895
id_10    0.873123
dtype: float64

In [8]:
# 5. Decide columns to drop (same for train & test)
def get_useless_columns(df, na_threshold=0.8):
    na_ratio = df.isna().mean()
    cols_high_na = na_ratio[na_ratio > na_threshold].index.tolist()
    
    nunique = df.nunique(dropna=False)
    cols_constant = nunique[nunique <= 1].index.tolist()
    
    drop_cols = list(set(cols_high_na + cols_constant))
    return drop_cols

# Note: exclude target from dropping
train_no_target = train.drop(columns=[target_col])
drop_cols = get_useless_columns(train_no_target, na_threshold=0.8)

print("Columns to drop (count):", len(drop_cols))

Columns to drop (count): 74


In [10]:
# 6. Drop columns safely from train & test
cols_train = [c for c in drop_cols if c in train.columns]
cols_test = [c for c in drop_cols if c in test.columns]

train_clean = train.drop(columns=cols_train)
test_clean  = test.drop(columns=cols_test)

print("Dropped columns from train:", len(cols_train))
print("Dropped columns from test:", len(cols_test))
print("Train clean shape:", train_clean.shape)
print("Test clean shape :", test_clean.shape)


Dropped columns from train: 74
Dropped columns from test: 55
Train clean shape: (590540, 360)
Test clean shape : (506691, 378)


In [12]:
# 7. Save intermediate cleaned merged data
train_clean.to_csv("../data/train_merged_clean.csv", index=False)
test_clean.to_csv("../data/test_merged_clean.csv", index=False)

print("Saved train_merged_clean.csv and test_merged_clean.csv")


Saved train_merged_clean.csv and test_merged_clean.csv
